# Optional Embedding History - From Counts to Static Geometry

**Guiding question:** what information did each representation add, and which ambiguity remained impossible until contextual models?

> **What you already know:** token IDs address trainable rows in `nn.Embedding`.
> **The gap this chapter closes:** why Word2Vec and GloVe were important steps beyond counts, and why both remain static.
> **What you'll have by the end:** measured local-prediction and global-co-occurrence geometry plus the `bank` boundary.
> **What this chapter is not:** sentiment classification, a pretrained pipeline demo, or semantic search.


In [ ]:
# -- Setup and one running corpus -------------------------------------------------
from collections import Counter
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt

torch.manual_seed(5)
np.random.seed(5)
plt.rcParams.update({"figure.facecolor": "#1a1a2e", "axes.facecolor": "#1a1a2e"})

sentences = [
    "river bank water stream",
    "river shore water bank",
    "bank loan money credit",
    "loan lender credit bank",
    "king queen royal palace",
    "man woman human person",
    "king man royal",
    "queen woman royal",
]
tokenized = [sentence.split() for sentence in sentences]
words = sorted({word for sentence in tokenized for word in sentence})
word_to_id = {word: index for index, word in enumerate(words)}
print(f"Vocabulary: {len(words)} words")


## 1 - Counts know presence, not learned similarity

```mermaid
flowchart LR
    A["one-hot word identity"] --> B["bag-of-words sentence counts"]
    B --> C["fixed sparse axes"]
    C --> D["no learned geometry"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- One-hot and bag-of-words baseline --------------------------------------------
one_hot = torch.eye(len(words))
first_bow = torch.zeros(len(words))
for token in tokenized[0]:
    first_bow[word_to_id[token]] += 1
river_bank_cosine = nn.functional.cosine_similarity(
    one_hot[word_to_id["river"]].unsqueeze(0),
    one_hot[word_to_id["bank"]].unsqueeze(0),
).item()
print(f"One-hot cosine(river, bank): {river_bank_cosine:.1f}")
print(f"First sentence BOW nonzero entries: {int((first_bow > 0).sum())}")
print("Takeaway: counts preserve occurrence; different word axes stay orthogonal.")


## 2 - Word2Vec learns from local prediction

The skip-gram complaint is concrete: words that predict similar nearby words should move toward useful geometry.


In [ ]:
# -- Build local center-context pairs ---------------------------------------------
pairs = []
for sentence in tokenized:
    for center_index, center in enumerate(sentence):
        for context_index in range(max(0, center_index - 1), min(len(sentence), center_index + 2)):
            if center_index != context_index:
                pairs.append((word_to_id[center], word_to_id[sentence[context_index]]))
centers = torch.tensor([pair[0] for pair in pairs])
contexts = torch.tensor([pair[1] for pair in pairs])

class SkipGram(nn.Module):
    def __init__(self, vocabulary_size, dimension=8):
        super().__init__()
        self.input = nn.Embedding(vocabulary_size, dimension)
        self.output = nn.Linear(dimension, vocabulary_size, bias=False)

    def forward(self, ids):
        return self.output(self.input(ids))

word2vec = SkipGram(len(words))
optimizer = torch.optim.Adam(word2vec.parameters(), lr=0.05)
losses = []
for _ in range(500):
    logits = word2vec(centers)
    loss = nn.functional.cross_entropy(logits, contexts)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach()))

w2v = word2vec.input.weight.detach()
print(f"Skip-gram loss: {losses[0]:.4f} -> {losses[-1]:.4f}")


In [ ]:
# -- Static geometry and analogy-style vector arithmetic -------------------------
def nearest(vector, matrix, excluded=()):
    scores = nn.functional.cosine_similarity(vector.unsqueeze(0), matrix)
    for index in excluded:
        scores[index] = -torch.inf
    return int(scores.argmax()), float(scores.max())

analogy = w2v[word_to_id["king"]] - w2v[word_to_id["man"]] + w2v[word_to_id["woman"]]
analogy_id, analogy_score = nearest(
    analogy,
    w2v,
    excluded=(word_to_id["king"], word_to_id["man"], word_to_id["woman"]),
)
print(f"king - man + woman -> {words[analogy_id]!r} (cosine={analogy_score:.3f})")
print("Boundary: a tiny corpus makes this evidence illustrative, not a production benchmark.")


## 3 - GloVe learns from global co-occurrence ratios

```mermaid
flowchart LR
    A["whole corpus"] --> B["co-occurrence matrix"]
    B --> C["weighted factorization objective"]
    C --> D["static word vectors"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Global co-occurrence matrix and compact GloVe objective ---------------------
cooccurrence = torch.zeros(len(words), len(words))
for sentence in tokenized:
    ids = [word_to_id[word] for word in sentence]
    for center_index, center_id in enumerate(ids):
        for context_index, context_id in enumerate(ids):
            if center_index != context_index:
                cooccurrence[center_id, context_id] += 1 / abs(center_index - context_index)

dimension = 8
main = nn.Embedding(len(words), dimension)
context = nn.Embedding(len(words), dimension)
main_bias = nn.Embedding(len(words), 1)
context_bias = nn.Embedding(len(words), 1)
parameters = list(main.parameters()) + list(context.parameters()) + list(main_bias.parameters()) + list(context_bias.parameters())
optimizer = torch.optim.Adam(parameters, lr=0.05)
rows, columns = torch.where(cooccurrence > 0)
counts = cooccurrence[rows, columns]
weights = torch.clamp((counts / 10.0).pow(0.75), max=1.0)
glove_losses = []

for _ in range(600):
    prediction = (
        (main(rows) * context(columns)).sum(dim=1)
        + main_bias(rows).squeeze(1)
        + context_bias(columns).squeeze(1)
    )
    loss = (weights * (prediction - counts.log()).pow(2)).mean()
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    glove_losses.append(float(loss.detach()))

glove = (main.weight + context.weight).detach()
print(f"GloVe-style loss: {glove_losses[0]:.4f} -> {glove_losses[-1]:.4f}")


In [ ]:
# -- Compare static neighborhoods under two training signals ---------------------
def neighbors(word, matrix, count=3):
    index = word_to_id[word]
    scores = nn.functional.cosine_similarity(matrix[index].unsqueeze(0), matrix)
    scores[index] = -torch.inf
    values, indices = torch.topk(scores, count)
    return [(words[i], round(float(value), 3)) for i, value in zip(indices, values)]

for target in ("river", "loan", "king"):
    print(f"{target:>5} Word2Vec: {neighbors(target, w2v)}")
    print(f"{target:>5} GloVe:    {neighbors(target, glove)}")
print("Takeaway: local prediction and global counts create different static geometry.")


## 4 - The static/contextual boundary

Both methods learn one row for `bank`.

```mermaid
flowchart LR
    B["one static bank row"] --> R["river context mixing"]
    B --> L["loan context mixing"]
    R --> BR["river-bank token representation"]
    L --> BL["loan-bank token representation"]
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style L fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style BR fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style BL fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- One static row, two contextualized toy representations ----------------------
bank = glove[word_to_id["bank"]]
river_bank = bank + glove[word_to_id["river"]]
loan_bank = bank + glove[word_to_id["loan"]]
contextual_difference = torch.dist(river_bank, loan_bank).item()
print(f"Static bank row count: 1")
print(f"Toy contextual representation distance: {contextual_difference:.4f}")
print("Boundary: neighbor mixing illustrates the need; Transformers implement the real contextual mechanism.")


## 5 - Coverage ledger

**Built and measured:** one-hot/BOW, skip-gram Word2Vec, GloVe-style global objective, analogies, static neighborhoods.

**Explained or illustrated:** local versus global training signals and the static/contextual boundary.

**Named and out of scope:** sentiment classification, pretrained pipeline demos, and semantic search. Semantic search belongs in RAG, where retrieval quality can be measured against a real job.
